In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv("/mnt/c/Users/rajpa/MachineLearning/Data_Folder/titanic.csv")
df.columns = df.columns.str.strip()
df.columns = df.columns.str.lower()
df = df.drop(columns='cabin')

In [16]:
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
df = df.dropna()

In [17]:
df['familysize'] = df['sibsp'] + df['parch']
df = df.drop(columns=['sibsp','parch'])
df['sex'] = df['sex'].map({'male':0,'female':1})

df = pd.get_dummies(df,columns=['embarked'],drop_first=True)

In [18]:
cols_scale = ['age','fare','familysize']

for col in cols_scale:
    min_val = df[col].min()
    max_val = df[col].max()
    df[col] = (df[col] - min_val) / (max_val - min_val)

In [19]:
features = ['pclass','sex','age','fare','familysize','embarked_Q','embarked_S']
X = df[features].values
y = df['survived'].values

X = X.astype(float)
y = y.astype(float)

In [33]:
class MyLogisticRegression:
    def __init__(self,learning_rate,iterations):
        self.lr = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None
        self.cost_history = []
    def _sigmoid(self,z):
        if not hasattr(np, "exp"):
            raise TypeError("np is not NumPy! Did you overwrite it?")
        return 1 / (1 + np.exp(-z))

    def fit(self,X,y):
        n_samples,n_features = X.shape
        y = y.reshape(-1,1)

        self.weights = np.zeros((n_features,1))
        self.bias = 0

        for i in range(self.iterations):
            linear_model = np.dot(X,self.weights) + self.bias

            y_pred = self._sigmoid(linear_model)

            epsilon = 1e-15
            y_pred_clipped = np.clip(y_pred,epsilon,1-epsilon)
            cost = -(1/n_samples) * np.sum(y*np.log(y_pred_clipped)+(1-y) * np.log(1-y_pred_clipped))
            self.cost_history.append(cost)

            error = y_pred - y
            dw = (1/n_samples) * np.dot(X.T, error)
            db = (1/n_samples) * np.sum(error)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            if i % 1000 == 0:
                print(f"Iter {i}: Cost {cost:.4f}")
    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self._sigmoid(linear_model)
        return [1 if i > 0.5 else 0 for i in y_predicted]            
        

In [34]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [35]:
print(f"Training on {len(X_train)} passengers...")

Training on 711 passengers...


Training the model

In [38]:
model = MyLogisticRegression(learning_rate=0.1,iterations=5000)
model.fit(X_train,y_train)

Iter 0: Cost 0.6931
Iter 1000: Cost 0.4582
Iter 2000: Cost 0.4514
Iter 3000: Cost 0.4472
Iter 4000: Cost 0.4447


Evaluating the model



In [39]:
predictions = model.predict(X_test)

In [40]:
accuracy = np.mean(predictions == y_test.flatten())
print(f"\nAccuracy on Test Set: {accuracy * 100:.2f}%")


Accuracy on Test Set: 78.65%


In [41]:
from sklearn.linear_model import LogisticRegression
sk_model = LogisticRegression()
sk_model.fit(X_train, y_train.flatten())
sk_acc = sk_model.score(X_test, y_test.flatten())
print(f"Sklearn Accuracy: {sk_acc * 100:.2f}%")

if abs(accuracy - sk_acc) < 0.05:
    print("SUCCESS: You matched the industry standard.")
else:
    print("FAIL: Check your learning rate or normalization.")

Sklearn Accuracy: 79.21%
SUCCESS: You matched the industry standard.


In [42]:
print(model.weights)

[[-1.02975306]
 [ 2.69797789]
 [-2.13516701]
 [ 0.54163975]
 [-1.49096838]
 [-0.15116228]
 [-0.52388091]]
